In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 15, 'lines.linewidth': 2,
    'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 1200,
})

import yaml
import numpy as np

import os
import sys
sys.path.append(f'{os.getcwd()}/irc_gym')

from irc.manager import IRCManager
from auditoryforage.utils import plot_AF_episode

# Train one agent for a single environment

In [ ]:
defaults = {
    'agent.env._target_': 'auditoryforage.AF_env.AuditoryForaging',
    'agent.model._target_': 'irc.model.FuncBeliefModel',
}
manager = IRCManager(defaults=defaults)

## Train an agent
We train a rational agent for the assume environment parameter [$lick\_cost, food\_reward, attention\_cost\_coeff, attention\_cost\_temp, penalty\_cost, iti\_cost$].

In [ ]:
env_param = [-3.0, 12.0, 15, 10, -8, -4]
num_epochs = 50

#Lokesh - added this for recent SB3
seed = 100

#Lokesh - added this for recent SB3
agent = manager.train_agent(env_param, num_epochs=num_epochs)
# agent = manager.train_agent(env_param, seed=seed, num_epochs=num_epochs)

agent, fig = manager.inspect_agent(env_param, figsize=(5, 2.5))

## Run the agent in an environment
We create another environment which has the same observation space and action space as the assumed one, albeit with a different set of environment parameters.

In [ ]:
from auditoryforage.AF_env import AuditoryForaging

# Change below line if you want to try the trained model on a different set of environemnt.
# env_param = [-3.0, 8.0, 13, 8, -8, -4]

env = AuditoryForaging(spec={'experiment': {'prob_01': 0.5},'agent':{'lick_cost':env_param[0],'food_reward':env_param[1],'attention_cost_coeff':env_param[2], 'attention_cost_temp': env_param[3], 'penalty_cost': env_param[4], 'iti_cost': env_param[5]}})
episode = agent.run_one_episode(env=env, num_steps=60, q_states = [[i] for i in range(env.no_nodes)])
fig = plot_AF_episode(episode, env)

env.spec



In [ ]:
# agent1, fig = manager.inspect_agent(env_param, figsize=(5, 2.5), seed = 0)
# agent2, fig = manager.inspect_agent(env_param, figsize=(5, 2.5), seed = 1)
# agent3, fig = manager.inspect_agent(env_param, figsize=(5, 2.5), seed = 2)

# beliefs = episode['beliefs']

# action_distributions1 = agent1.agent_action_distribution(beliefs)
# action_distributions2 = agent2.agent_action_distribution(beliefs)
# action_distributions3 = agent3.agent_action_distribution(beliefs)

In [22]:
env_param = [-3.0, 12.0, 15, 10, -8, -4]
beliefs = episode['beliefs']

attention_cost_coeff_list = [1, 50, 100] # attention_cost_coeff
replace_index = 2

num_epochs = 50
agents_list = []
action_distributions_list = []

for attention_cost_coeff in attention_cost_coeff_list:
    env_param_temp = env_param[:replace_index]+[attention_cost_coeff]+env_param[replace_index+1:]
    print(env_param_temp)
    agents_list.append(manager.train_agent(env_param_temp, num_epochs=num_epochs))
    action_distributions_list.append(agent.agent_action_distribution(beliefs))


[-3.0, 12.0, 1, 10, -8, -4]
Checkpoint (epoch 22) loaded.

Epoch: 23/50
Agent trained by 256 time steps (0m02.14s).

Epoch: 24/50
Agent trained by 256 time steps (0m02.11s).
Agent evaluated for 12 episodes of length 20. Average return -2.38. (0m00.10s)

Epoch: 25/50
Agent trained by 256 time steps (0m02.16s).

Epoch: 26/50
Agent trained by 256 time steps (0m02.16s).
Agent evaluated for 12 episodes of length 20. Average return -0.64. (0m00.11s)

Epoch: 27/50
Agent trained by 256 time steps (0m02.15s).

Epoch: 28/50
Agent trained by 256 time steps (0m02.21s).
Agent evaluated for 12 episodes of length 20. Average return 0.66. (0m00.13s)

Epoch: 29/50
Agent trained by 256 time steps (0m02.13s).

Epoch: 30/50
Agent trained by 256 time steps (0m02.01s).
Agent evaluated for 12 episodes of length 20. Average return 0.94. (0m00.10s)

Epoch: 31/50
Agent trained by 256 time steps (0m02.03s).

Epoch: 32/50
Agent trained by 256 time steps (0m02.08s).
Agent evaluated for 12 episodes of length 20. Av

In [ ]:
reference_dists = np.array(action_distributions_list[0])
difference = []
for action_distributions in action_distributions_list:
    difference.append(np.sum((np.array(action_distributions)- reference_dists)**2))
plt.plot(difference)
plt.show()

In [ ]:
np.sum((np.array(action_distributions_list[0])- np.array(action_distributions_list[2]))**2)

In [ ]:
np.array(action_distributions_list[0])- np.array(action_distributions_list[2])